# Meta-Learner Comparison for Stacking Ensemble
## Energy Theft Detection — Binary Classification (Normal vs Theft)

---

### Purpose of this notebook
After building the stacking ensemble (RF + XGB + MLP as base models), we need to **select the best meta-learner** in a methodologically sound way. This notebook:

1. Generates **Out-of-Fold (OOF) meta-features** from the base models on training data only
2. Trains and evaluates **5 candidate meta-learners** using cross-validation (no test set leakage)
3. Compares them across **5 criteria**: CV Accuracy, Calibration (ECE), Training Time, Interpretability, Stability
4. Runs **Friedman + Nemenyi statistical tests** to identify which differences are real
5. Produces a **final scorecard table** for thesis use
6. Evaluates the winning meta-learner on the **test set** (done only once, at the very end)

### Binary classification: Normal (0) and Theft (1)

---

### Key methodological rule followed:
> Meta-learner selection is performed **exclusively on training data via cross-validation**. The test set is only used once, for the final evaluation of the selected model. This prevents test set leakage and ensures honest reporting.

---
## CELL 1 — Install Required Libraries
Run this cell first if any packages are missing in your environment.

In [ ]:
# Uncomment and run if packages are not already installed
!pip install scikit-learn xgboost scikit-posthocs pandas numpy matplotlib seaborn scipy joblib

print('If no errors above, all packages are already installed.')
print('Proceed to Cell 2.')

---
## CELL 2 — Import All Libraries
Imports everything needed across the entire notebook up front.

In [ ]:
# ── Standard library ──────────────────────────────────────────────────────────
import time
import warnings
warnings.filterwarnings('ignore')

# ── Data ──────────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np

# ── Plotting ──────────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

# ── Sklearn: preprocessing ────────────────────────────────────────────────────
from sklearn.preprocessing import StandardScaler, LabelEncoder, label_binarize
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_val_score, cross_validate
)

# ── Sklearn: base models ──────────────────────────────────────────────────────
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier

# ── Sklearn: meta-learner candidates ─────────────────────────────────────────
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import ExtraTreesClassifier

# ── Sklearn: metrics ──────────────────────────────────────────────────────────
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, auc
)
from sklearn.calibration import calibration_curve, CalibratedClassifierCV

# ── Statistical tests ─────────────────────────────────────────────────────────
from scipy.stats import friedmanchisquare
try:
    import scikit_posthocs as sp
    POSTHOCS_AVAILABLE = True
    print('scikit-posthocs available — Nemenyi test will run.')
except ImportError:
    POSTHOCS_AVAILABLE = False
    print('scikit-posthocs not found. Install with: pip install scikit-posthocs')
    print('Friedman test will still run; Nemenyi post-hoc will be skipped.')

# ── Global settings ───────────────────────────────────────────────────────────
RANDOM_STATE = 42
TEST_SIZE    = 0.20
N_CV_FOLDS   = 3

pd.set_option('display.float_format', '{:.4f}'.format)
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

print('\nAll imports successful!')
print(f'Random state : {RANDOM_STATE}')
print(f'Test size    : {TEST_SIZE*100:.0f}%')
print(f'CV folds     : {N_CV_FOLDS}')

---
## CELL 3 — Load Dataset
Load the dataset. Update `DATASET_PATH` and `TARGET_COL` to match your file.
Expected classes: **Normal** and **Theft** (binary).

In [ ]:
# ── CONFIGURE THESE ───────────────────────────────────────────────────────────
DATASET_PATH      = "../data/Dataset_actual.csv"   # <-- UPDATE: path to your dataset
import os; os.makedirs('../results', exist_ok=True)
TARGET_COL        = 'theft'              # <-- UPDATE: name of your target column
CONSUMER_TYPE_COL = 'Class'      # <-- UPDATE: name of consumer type column (if any)
# ─────────────────────────────────────────────────────────────────────────────

df = pd.read_csv(DATASET_PATH)

print(f'Dataset shape : {df.shape}')
print(f'Columns       : {df.columns.tolist()}')
print(f'\nTarget distribution:')
print(df[TARGET_COL].value_counts())
print(f'\nMissing values: {df.isnull().sum().sum()} total')

---
## CELL 4 — Preprocessing
- Drop nulls
- Label-encode the consumer_type column if present (category codes)
- Label-encode the target (Normal=0, Theft=1)
- Standard scaling is applied later (after splitting) to prevent data leakage

In [ ]:
# Drop missing values
df.dropna(inplace=True)
print(f'Shape after dropping NaNs: {df.shape}')

# Encode consumer type if present (categorical → integer codes)
if CONSUMER_TYPE_COL in df.columns:
    df[CONSUMER_TYPE_COL] = df[CONSUMER_TYPE_COL].astype('category').cat.codes

# Encode target labels (Normal → 0, Theft → 1)
le = LabelEncoder()
df[TARGET_COL] = le.fit_transform(df[TARGET_COL])

# Identify feature columns
all_feature_cols = [c for c in df.columns if c != TARGET_COL]

print(f'\nAll feature cols ({len(all_feature_cols)}): {all_feature_cols}')
print(f'Target classes after encoding: {dict(zip(le.classes_, le.transform(le.classes_)))}')
print(f'\nClass distribution after encoding:')
print(pd.Series(df[TARGET_COL]).value_counts().rename(index=dict(enumerate(le.classes_))))

---
## CELL 5 — Build Feature Matrix and Labels
Binary classification: **Normal (0)** vs **Theft (1)**

In [ ]:
X = df[all_feature_cols].values
y = df[TARGET_COL].values
class_names = [str(c) for c in le.classes_]  # e.g. ['Normal', 'Theft']
n_classes   = len(class_names)

print(f'X shape    : {X.shape}')
print(f'y shape    : {y.shape}')
print(f'Classes    : {class_names}')
print(f'n_classes  : {n_classes}')
print(f'Class counts: {dict(zip(class_names, np.bincount(y)))}')

---
## CELL 6 — Train/Test Split and Standard Scaling
80% train / 20% test. Scaler is **fit on train only** to prevent leakage.

In [ ]:
def split_and_scale(X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE):
    """Stratified split + StandardScaler fit on train only."""
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=y
    )
    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X_tr)
    X_te = scaler.transform(X_te)
    return X_tr, X_te, y_tr, y_te, scaler

X_train, X_test, y_train, y_test, scaler = split_and_scale(X, y)

print('=== Train/Test Split ===')
print(f'  Train: {X_train.shape}  |  Test: {X_test.shape}')
print(f'  Train class dist: {dict(zip(class_names, np.bincount(y_train)))}')
print(f'  Test  class dist: {dict(zip(class_names, np.bincount(y_test)))}')

---
## CELL 7 — Define Base Models
Three base models: RF, XGB, MLP. These are fixed and do **not** change across meta-learner experiments.

In [ ]:
def get_base_models():
    """
    Returns the three base models.
    RF  : Random Forest   — 100 trees, Gini
    XGB : XGBoost         — max_depth=6, eta=0.3
    MLP : Multi-layer Perceptron — hidden=(100,), ReLU, Adam
    """
    rf = RandomForestClassifier(
        n_estimators=50, criterion='gini',
        random_state=0, min_samples_split=2, min_samples_leaf=1,
        n_jobs=-1
    )
    xgb = XGBClassifier(
        max_depth=6, learning_rate=0.3, scale_pos_weight=1,
        min_child_weight=1, booster='gbtree',
        eval_metric='logloss',
        random_state=RANDOM_STATE, verbosity=0,
        n_jobs=-1
    )
    mlp = MLPClassifier(
        hidden_layer_sizes=(100,), activation='relu',
        solver='adam', alpha=0.0001,
        max_iter=200, random_state=RANDOM_STATE
    )
    return [('RF', rf), ('XGB', xgb), ('MLP', mlp)]

print('Base models defined:')
for name, _ in get_base_models():
    print(f'  • {name}')
print('\nThese are fixed across ALL meta-learner experiments.')

---
## CELL 8 — Define All Meta-Learner Candidates

Five meta-learners are compared (Gradient Boosting excluded):

| # | Meta-Learner | Why included |
|---|---|---|
| 1 | Logistic Regression | Baseline; interpretable; well-calibrated |
| 2 | Ridge Classifier | Regularized linear; fast; less sensitive to multicollinearity |
| 3 | Decision Tree | Interpretable; non-linear boundaries |
| 4 | SVM (RBF kernel) | Strong non-linear separator; good with probability features |
| 5 | Extra Trees | Fast ensemble; high variance base models stabilised by averaging |

In [ ]:
def get_meta_learner_candidates():
    """
    Returns dict of {name: (model, is_interpretable)}.
    is_interpretable = True if model provides coefficients or feature importances.
    Gradient Boosting is excluded.
    """
    candidates = {
        'Logistic Regression': (
            LogisticRegression(
                max_iter=1000, solver='lbfgs', C=1.0,
                random_state=RANDOM_STATE
            ),
            True   # interpretable via coefficients
        ),
        'Ridge Classifier': (
            CalibratedClassifierCV(
                RidgeClassifier(alpha=1.0, random_state=RANDOM_STATE),
                method='isotonic', cv=3
            ),
            True   # interpretable via wrapped coef_
        ),
        'Decision Tree': (
            DecisionTreeClassifier(
                max_depth=5, random_state=RANDOM_STATE
            ),
            True   # interpretable via feature importance + tree structure
        ),
        
        'Extra Trees': (
            ExtraTreesClassifier(
                n_estimators=100, random_state=RANDOM_STATE,
                n_jobs=-1
            ),
            False  # black box (feature importance only)
        ),
    }
    return candidates

print('Meta-learner candidates (5 total — Gradient Boosting excluded):')
for i, (name, (_, interp)) in enumerate(get_meta_learner_candidates().items(), 1):
    interp_str = '✓ Interpretable' if interp else '✗ Black box'
    print(f'  {i}. {name:<25} {interp_str}')

---
## CELL 9 — Generate Out-of-Fold (OOF) Meta-Features

**What this does and why it matters:**

Each base model is trained on k-1 folds and predicts on the held-out fold. This produces **out-of-fold probability predictions** — unbiased estimates of how the base model would perform on unseen data. These OOF predictions become the input features for the meta-learner.

This is done on **training data only**. The test set is not touched here.

OOF matrix shape: `(n_train_samples, n_base_models × n_classes)`
- Binary: `(n_train, 3 × 2)` = `(n_train, 6)`

In [ ]:
def generate_oof_meta_features(X_train, y_train, n_classes, n_folds=N_CV_FOLDS):
    """
    Generates Out-of-Fold probability predictions from all base models.
    Also trains each base model on the full training set for later
    generation of test-set meta-features.

    Returns:
        oof_matrix      : (n_train, n_base_models * n_classes) array
        trained_bases   : list of fully-trained base models (on full train set)
    """
    base_models    = get_base_models()
    n_base         = len(base_models)
    n_train        = X_train.shape[0]
    oof_matrix     = np.zeros((n_train, n_base * n_classes))
    skf            = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=RANDOM_STATE)

    print(f'  Generating OOF meta-features using {n_folds}-fold CV...')

    for b_idx, (name, model) in enumerate(base_models):
        print(f'    Base model {b_idx+1}/{n_base}: {name}')
        col_start = b_idx * n_classes
        col_end   = col_start + n_classes

        for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
            X_tr_fold, X_val_fold = X_train[tr_idx], X_train[val_idx]
            y_tr_fold             = y_train[tr_idx]
            from sklearn.base import clone
            fold_model = clone(model)
            fold_model.fit(X_tr_fold, y_tr_fold)
            oof_matrix[val_idx, col_start:col_end] = fold_model.predict_proba(X_val_fold)

    # Train each base model on FULL training set (for test-set meta-features later)
    trained_bases = []
    print('  Training base models on full training set...')
    for name, model in base_models:
        model.fit(X_train, y_train)
        trained_bases.append((name, model))

    print(f'  OOF matrix shape: {oof_matrix.shape}')
    return oof_matrix, trained_bases


def generate_test_meta_features(X_test, trained_bases, n_classes):
    """
    Uses fully-trained base models to generate meta-features for the test set.
    Only called at the very end for final evaluation.
    """
    n_base       = len(trained_bases)
    test_matrix  = np.zeros((X_test.shape[0], n_base * n_classes))
    for b_idx, (name, model) in enumerate(trained_bases):
        col_start = b_idx * n_classes
        col_end   = col_start + n_classes
        test_matrix[:, col_start:col_end] = model.predict_proba(X_test)
    return test_matrix


# ── Run OOF generation ────────────────────────────────────────────────────────
print('Generating OOF meta-features...')
oof, trained_bases = generate_oof_meta_features(X_train, y_train, n_classes)

print('\nOOF meta-feature generation complete.')
print(f'  OOF shape: {oof.shape}  (= n_train × [3 base models × {n_classes} classes])')

---
## CELL 10 — Define ECE (Expected Calibration Error) Calculator

**What ECE measures:**
Calibration is the agreement between a model's predicted confidence and actual accuracy. If a model says "80% confident" for 100 predictions, ~80 should be correct.

ECE = weighted average of |predicted confidence − actual accuracy| across probability bins.
**Lower ECE = better calibrated.** For a decision support system, calibration is critical — operators act on confidence scores.

In [ ]:
def compute_ece(y_true, y_prob, n_bins=10):
    """
    Computes Expected Calibration Error (ECE).
    Uses the confidence of the predicted class (max probability).

    ECE = Σ (|bin| / N) × |accuracy(bin) − confidence(bin)|

    Args:
        y_true : true class labels (1D array)
        y_prob : predicted probabilities (2D array, shape n_samples × n_classes)
        n_bins : number of calibration bins (default 10)

    Returns:
        ece (float) — lower is better
    """
    confidences = np.max(y_prob, axis=1)
    predictions = np.argmax(y_prob, axis=1)
    correct     = (predictions == y_true).astype(float)

    bin_edges = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    n   = len(y_true)

    for i in range(n_bins):
        in_bin = (confidences > bin_edges[i]) & (confidences <= bin_edges[i + 1])
        if in_bin.sum() == 0:
            continue
        bin_acc  = correct[in_bin].mean()
        bin_conf = confidences[in_bin].mean()
        ece     += (in_bin.sum() / n) * abs(bin_acc - bin_conf)

    return ece


print('ECE function defined.')
print('Lower ECE = better calibration = more trustworthy confidence scores.')

---
## CELL 11 — Run Meta-Learner Comparison (CV on OOF Features)

For each meta-learner candidate, using only the OOF training matrix:
- **CV Accuracy**: 5-fold stratified CV mean ± std
- **CV F1 (binary)**: 5-fold binary F1 mean
- **Stability**: std of CV accuracy across folds (lower = more stable)
- **Training time**: time to fit on full OOF training matrix
- **ECE**: calibration error on OOF held-out predictions
- **Interpretable**: whether the model exposes coefficients/feature importance

⚠️ **No test set is used in this cell.**

In [ ]:
def compare_meta_learners(oof_X, y_train, scenario_name, n_folds=N_CV_FOLDS):
    """
    Evaluates all meta-learner candidates on OOF meta-features using CV.
    Returns a results dict and a per-fold accuracy matrix (for statistical tests).
    """
    candidates       = get_meta_learner_candidates()
    skf              = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=RANDOM_STATE)
    results          = {}
    fold_scores_all  = {}   # {meta_name: [fold1_acc, fold2_acc, ...]}

    print(f'\n{"="*60}')
    print(f'  META-LEARNER COMPARISON — {scenario_name}')
    print(f'{"="*60}')

    for meta_name, (meta_model, is_interpretable) in candidates.items():
        print(f'  Evaluating: {meta_name}...')
        from sklearn.base import clone

        fold_accs  = []
        fold_f1s   = []
        ece_scores = []

        for tr_idx, val_idx in skf.split(oof_X, y_train):
            X_tr, X_val = oof_X[tr_idx], oof_X[val_idx]
            y_tr, y_val = y_train[tr_idx], y_train[val_idx]

            m = clone(meta_model)
            m.fit(X_tr, y_tr)

            y_pred = m.predict(X_val)
            y_prob = m.predict_proba(X_val)

            fold_accs.append(accuracy_score(y_val, y_pred))
            fold_f1s.append(f1_score(y_val, y_pred, average='binary', zero_division=0))
            ece_scores.append(compute_ece(y_val, y_prob))

        # Training time on full OOF matrix
        m_full = clone(meta_model)
        t0     = time.time()
        m_full.fit(oof_X, y_train)
        train_time = time.time() - t0

        results[meta_name] = {
            'CV Accuracy Mean':  np.mean(fold_accs) * 100,
            'CV Accuracy Std':   np.std(fold_accs)  * 100,
            'CV F1 Binary Mean': np.mean(fold_f1s)  * 100,
            'ECE (↓ better)':    np.mean(ece_scores),
            'Train Time (s)':    train_time,
            'Interpretable':     '✓' if is_interpretable else '✗',
            'Stability (std↓)':  np.std(fold_accs) * 100,
            'trained_model':     m_full
        }
        fold_scores_all[meta_name] = fold_accs

        print(f'    CV Acc: {np.mean(fold_accs)*100:.2f}% ± {np.std(fold_accs)*100:.2f}%  '
              f'| F1: {np.mean(fold_f1s)*100:.2f}%  '
              f'| ECE: {np.mean(ece_scores):.4f}  '
              f'| Time: {train_time:.2f}s')

    return results, fold_scores_all


# ── Run comparison ────────────────────────────────────────────────────────────
results, fold_scores = compare_meta_learners(oof, y_train, 'Binary (Normal vs Theft)')

---
## CELL 12 — Friedman Statistical Test

**Why Friedman and not ANOVA?**
ANOVA assumes normality and equal variance — not guaranteed with classifier accuracies. The Friedman test is non-parametric and designed exactly for comparing multiple classifiers across multiple folds.

- **H₀**: All meta-learners perform equally (no significant difference)
- **H₁**: At least one meta-learner differs significantly
- If **p < 0.05**, reject H₀ → differences are real → run Nemenyi post-hoc

In [ ]:
def run_friedman_test(fold_scores_dict, scenario_name):
    """
    Runs Friedman test across all meta-learners' per-fold CV accuracy scores.
    fold_scores_dict: {meta_name: [fold1_acc, fold2_acc, ...]}
    """
    names  = list(fold_scores_dict.keys())
    scores = [fold_scores_dict[n] for n in names]

    stat, p = friedmanchisquare(*scores)

    print(f'\n{"─"*55}')
    print(f'  Friedman Test — {scenario_name}')
    print(f'{"─"*55}')
    print(f'  Chi-square statistic : {stat:.4f}')
    print(f'  p-value              : {p:.6f}')

    if p < 0.05:
        print(f'  Result  : ✓ SIGNIFICANT (p < 0.05)')
        print(f'  Meaning : At least one meta-learner is statistically different.')
        print(f'  Action  : Proceed to Nemenyi post-hoc test to identify which pairs differ.')
    else:
        print(f'  Result  : ✗ NOT significant (p ≥ 0.05)')
        print(f'  Meaning : No statistically significant difference between meta-learners.')
        print(f'  Action  : Select based on secondary criteria (ECE, speed, interpretability).')

    return p, names, scores


p_val, names_list, scores_list = run_friedman_test(fold_scores, 'Binary (Normal vs Theft)')

---
## CELL 13 — Nemenyi Post-Hoc Test (if Friedman is significant)

The Friedman test tells us *some* difference exists but not *which* pairs differ. The **Nemenyi test** does pairwise comparisons with multiple-comparison correction.

- Values in the matrix are **p-values** for each pair of meta-learners
- **p < 0.05** → the two meta-learners are statistically different
- **p ≥ 0.05** → the two are statistically indistinguishable (use other criteria to choose)

In [ ]:
def run_nemenyi_test(fold_scores_dict, p_friedman, scenario_name):
    """
    Runs Nemenyi post-hoc test if Friedman test was significant.
    Plots a heatmap of pairwise p-values.
    """
    if not POSTHOCS_AVAILABLE:
        print('scikit-posthocs not installed — skipping Nemenyi test.')
        print('Install with: pip install scikit-posthocs')
        return None

    if p_friedman >= 0.05:
        print(f'Friedman test was not significant for {scenario_name}.')
        print('Nemenyi test not applicable — differences are not statistically real.')
        return None

    names  = list(fold_scores_dict.keys())
    data   = np.array([fold_scores_dict[n] for n in names]).T  # shape: (n_folds, n_models)
    df_nemenyi = pd.DataFrame(data, columns=names)

    nemenyi_p = sp.posthoc_nemenyi_friedman(df_nemenyi)

    print(f'\nNemenyi Post-Hoc Test — {scenario_name}')
    print('p-value matrix (< 0.05 = significantly different pair):')

    # Plot heatmap
    fig, ax = plt.subplots(figsize=(9, 7))
    sns.heatmap(
        nemenyi_p, annot=True, fmt='.3f', cmap='RdYlGn',
        vmin=0, vmax=0.1, center=0.05,
        xticklabels=names, yticklabels=names,
        linewidths=0.5, ax=ax
    )
    ax.set_title(
        f'Nemenyi Post-Hoc p-values — {scenario_name}\n'
        f'(Green = significantly different | Red = not significantly different)',
        fontsize=11
    )
    plt.xticks(rotation=30, ha='right', fontsize=9)
    plt.yticks(rotation=0, fontsize=9)
    plt.tight_layout()
    plt.savefig(f'../results/nemenyi_{scenario_name.replace(" ","_")}.png', dpi=150, bbox_inches='tight')
    plt.show()

    return nemenyi_p


nemenyi_result = run_nemenyi_test(fold_scores, p_val, 'Binary_Normal_vs_Theft')

---
## CELL 14 — Calibration Reliability Diagrams

A reliability diagram plots **mean predicted probability vs actual fraction of positives** per bin. A perfectly calibrated model follows the diagonal (y = x).

Plotted using pre-computed ECE values from Cell 11.

In [ ]:
def plot_calibration_summary(results_dict):
    """
    Plots ECE comparison bar chart using already-computed ECE values.
    Avoids re-running CV — uses results stored from Cell 11.
    """
    fig, ax = plt.subplots(figsize=(10, 5))

    names  = list(results_dict.keys())
    eces   = [results_dict[n]['ECE (↓ better)'] for n in names]
    colors = plt.cm.Set2.colors
    bars   = ax.bar(names, eces, color=colors[:len(names)], edgecolor='black', alpha=0.8)
    ax.set_title('ECE (Calibration Error) — Binary Classification\nLower is better', fontsize=11)
    ax.set_ylabel('ECE')
    ax.set_xlabel('Meta-Learner')
    ax.set_xticklabels([n.replace(' ', '\n') for n in names], fontsize=9)
    ax.grid(True, axis='y', alpha=0.4)
    for bar, ece in zip(bars, eces):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                f'{ece:.4f}', ha='center', va='bottom', fontsize=8)

    plt.tight_layout()
    plt.savefig('../results/calibration_ece_summary.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Calibration summary plotted using pre-computed ECE values from Cell 11.')


plot_calibration_summary(results)

---
## CELL 15 — CV Accuracy Distribution (Box Plots)

Box plots show the distribution of per-fold accuracy for each meta-learner. A tighter box = more stable across folds. Outlier folds (dots outside whiskers) indicate instability.

In [ ]:
def plot_cv_boxplots(fold_scores_dict):
    """
    Box plots of per-fold CV accuracy for all meta-learners.
    """
    fig, ax = plt.subplots(figsize=(10, 6))

    names  = list(fold_scores_dict.keys())
    data   = [np.array(fold_scores_dict[n]) * 100 for n in names]

    bp = ax.boxplot(
        data, patch_artist=True, notch=False,
        medianprops={'color': 'black', 'linewidth': 2},
        whiskerprops={'linewidth': 1.5},
        capprops={'linewidth': 1.5},
        flierprops={'marker': 'o', 'markersize': 5, 'alpha': 0.5}
    )

    colors = plt.cm.Set2.colors
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.75)

    ax.set_xticks(range(1, len(names) + 1))
    ax.set_xticklabels([n.replace(' ', '\n') for n in names], fontsize=9)
    ax.set_title('CV Accuracy Distribution — Binary (Normal vs Theft)', fontsize=12)
    ax.set_ylabel('CV Accuracy (%)')
    ax.set_xlabel('Meta-Learner')
    ax.grid(True, axis='y', alpha=0.4)

    for i, d in enumerate(data):
        ax.text(i + 1, np.median(d) + 0.05, f'{np.median(d):.2f}%',
                ha='center', va='bottom', fontsize=7, fontweight='bold')

    plt.suptitle('Per-Fold CV Accuracy — All Meta-Learner Candidates', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../results/cv_boxplots.png', dpi=150, bbox_inches='tight')
    plt.show()


plot_cv_boxplots(fold_scores)

---
## CELL 16 — Build the Final Scorecard Table

This table combines all five evaluation criteria into one structured comparison for **thesis use**.

The criteria and their direction:
- **CV Accuracy** ↑ higher is better
- **ECE** ↓ lower is better (calibration)
- **Training Time** ↓ lower is better
- **Interpretable** — binary (✓/✗)
- **Stability (std)** ↓ lower is better

In [ ]:
def build_scorecard(results_dict, scenario_name):
    """
    Builds a clean scorecard DataFrame from results dict.
    Highlights best value per column.
    """
    rows = []
    for meta_name, vals in results_dict.items():
        rows.append({
            'Meta-Learner':        meta_name,
            'CV Accuracy (%) ↑':   round(vals['CV Accuracy Mean'], 2),
            'CV F1 Binary (%) ↑':  round(vals['CV F1 Binary Mean'], 2),
            'ECE ↓':               round(vals['ECE (↓ better)'], 4),
            'Train Time (s) ↓':    round(vals['Train Time (s)'], 3),
            'Stability Std (%) ↓': round(vals['Stability (std↓)'], 4),
            'Interpretable':       vals['Interpretable'],
        })

    df_score = pd.DataFrame(rows).set_index('Meta-Learner')

    print(f'\n{"="*70}')
    print(f'  SCORECARD — {scenario_name}')
    print(f'{"="*70}')

    styled = df_score.style \
        .highlight_max(subset=['CV Accuracy (%) ↑', 'CV F1 Binary (%) ↑'],
                       color='#c6efce') \
        .highlight_min(subset=['ECE ↓', 'Train Time (s) ↓', 'Stability Std (%) ↓'],
                       color='#c6efce') \
        .set_caption(f'Meta-Learner Scorecard — {scenario_name}') \
        .set_properties(**{'text-align': 'center'})

    display(styled)
    return df_score


scorecard = build_scorecard(results, 'Binary Classification (Normal vs Theft)')

---
## CELL 17 — Scorecard Radar (Spider) Chart

A radar chart gives a visual at-a-glance comparison across all 5 criteria simultaneously. Each axis is normalised 0–1 so that **outward always means better**.

In [ ]:
def plot_radar_chart(scorecard_df, scenario_name):
    """
    Radar chart comparing meta-learners across all scorecard criteria.
    All axes normalised so outward = better.
    """
    numeric_cols = ['CV Accuracy (%) ↑', 'CV F1 Binary (%) ↑',
                    'ECE ↓', 'Train Time (s) ↓', 'Stability Std (%) ↓']
    df_num = scorecard_df[numeric_cols].copy()

    df_norm = df_num.copy()
    for col in numeric_cols:
        col_min = df_num[col].min()
        col_max = df_num[col].max()
        if col_max == col_min:
            df_norm[col] = 0.5
            continue
        if '↓' in col:
            df_norm[col] = 1 - (df_num[col] - col_min) / (col_max - col_min)
        else:
            df_norm[col] = (df_num[col] - col_min) / (col_max - col_min)

    labels = ['CV\nAccuracy', 'CV\nF1 Binary',
              'Calibration\n(ECE inv)', 'Train\nSpeed', 'Stability']
    N = len(labels)
    angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
    angles += angles[:1]

    fig, ax = plt.subplots(figsize=(9, 9), subplot_kw=dict(polar=True))
    colors = plt.cm.tab10.colors

    for i, (meta_name, row) in enumerate(df_norm.iterrows()):
        values = row.tolist() + row.tolist()[:1]
        ax.plot(angles, values, 'o-', lw=2, color=colors[i], label=meta_name)
        ax.fill(angles, values, alpha=0.07, color=colors[i])

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(labels, fontsize=10)
    ax.set_ylim(0, 1)
    ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
    ax.set_yticklabels(['0.2', '0.4', '0.6', '0.8', '1.0'], fontsize=7)
    ax.set_title(f'Meta-Learner Scorecard Radar\n{scenario_name}',
                 fontsize=12, fontweight='bold', pad=20)
    ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15), fontsize=9)
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(f'../results/radar_{scenario_name.replace(" ","_")}.png', dpi=150, bbox_inches='tight')
    plt.show()


plot_radar_chart(scorecard, 'Binary Classification (Normal vs Theft)')

---
## CELL 18 — Select the Best Meta-Learner

Selection is based on the scorecard:
1. Highest CV accuracy (primary criterion)
2. Tie-breaking: lowest ECE, then lowest std, then interpretability preference

⚠️ The winning meta-learner is selected here. The test set is **only used in Cell 19**.

In [ ]:
def select_best_meta_learner(results_dict, scenario_name):
    """
    Selects best meta-learner from CV results.
    Primary: highest CV Accuracy
    Tiebreak: lowest ECE, then lowest std
    """
    best_name  = None
    best_acc   = -1
    best_ece   = 999
    best_std   = 999

    for name, vals in results_dict.items():
        acc = vals['CV Accuracy Mean']
        ece = vals['ECE (↓ better)']
        std = vals['Stability (std↓)']

        if (acc > best_acc or
           (abs(acc - best_acc) < 0.1 and ece < best_ece) or
           (abs(acc - best_acc) < 0.1 and abs(ece - best_ece) < 0.001 and std < best_std)):
            best_name = name
            best_acc  = acc
            best_ece  = ece
            best_std  = std

    print(f'\n{"─"*55}')
    print(f'  Selected Meta-Learner — {scenario_name}')
    print(f'{"─"*55}')
    print(f'  ✓  {best_name}')
    print(f'     CV Accuracy : {best_acc:.2f}%')
    print(f'     ECE         : {best_ece:.4f}')
    print(f'     Std (stab.) : {best_std:.4f}%')
    print(f'     Trained model retrieved from results dict.')

    return best_name, results_dict[best_name]['trained_model']


best_name, best_model = select_best_meta_learner(results, 'Binary Classification')

print(f'\nFinal selection:')
print(f'  Best meta-learner: {best_name}')

---
## CELL 19 — Final Test Set Evaluation (Done Once, for Selected Model Only)

This is the only cell that uses the test set. The selected meta-learner is evaluated on test-set meta-features generated by the fully-trained base models.

In [ ]:
def final_test_evaluation(best_meta_model, trained_bases,
                           X_test, y_test, n_classes, class_names, scenario_name):
    """
    Generates test-set meta-features and evaluates the selected meta-learner.
    Called only once.
    """
    # Step 1: Generate test meta-features using trained base models
    test_meta_X = generate_test_meta_features(X_test, trained_bases, n_classes)

    # Step 2: Predict with selected meta-learner
    y_pred = best_meta_model.predict(test_meta_X)
    y_prob = best_meta_model.predict_proba(test_meta_X)

    # Step 3: Compute all metrics
    acc   = accuracy_score(y_test, y_pred)  * 100
    prec  = precision_score(y_test, y_pred, average='binary', zero_division=0) * 100
    rec   = recall_score(y_test, y_pred, average='binary', zero_division=0) * 100
    f1    = f1_score(y_test, y_pred, average='binary', zero_division=0) * 100
    ece   = compute_ece(y_test, y_prob)
    try:
        auc_score = roc_auc_score(y_test, y_prob[:, 1]) * 100
    except Exception:
        auc_score = float('nan')

    print(f'\n{"═"*55}')
    print(f'  FINAL TEST RESULTS — {scenario_name}')
    print(f'{"═"*55}')
    print(f'  Accuracy   : {acc:.2f}%')
    print(f'  Precision  : {prec:.2f}%')
    print(f'  Recall     : {rec:.2f}%')
    print(f'  F1-Score   : {f1:.2f}%')
    print(f'  AUC-ROC    : {auc_score:.2f}%')
    print(f'  ECE        : {ece:.4f}')
    print()
    print('Per-class report:')
    print(classification_report(y_test, y_pred, target_names=class_names, zero_division=0))

    return {
        'accuracy': acc, 'precision': prec, 'recall': rec,
        'f1': f1, 'auc': auc_score, 'ece': ece,
        'y_pred': y_pred, 'y_prob': y_prob
    }


print('Running final test evaluation...')
print('(Test set is used for the first and only time below)\n')

final_results = final_test_evaluation(
    best_model, trained_bases,
    X_test, y_test, n_classes, class_names, 'Binary (Normal vs Theft)'
)

---
## CELL 20 — Confusion Matrix for Final Model

In [ ]:
def plot_confusion_matrix(y_true, y_pred, class_names, title):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(7, 5))
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues',
        xticklabels=class_names, yticklabels=class_names,
        linewidths=0.5
    )
    plt.title(f'Confusion Matrix — {title}', fontsize=12)
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.savefig(f'../results/cm_{title.replace(" ","_")}.png', dpi=150, bbox_inches='tight')
    plt.show()


plot_confusion_matrix(
    y_test, final_results['y_pred'], class_names,
    f'Best Stacking Model ({best_name}) — Binary'
)

---
## CELL 21 — ROC Curve for Final Model

In [ ]:
def plot_roc_curve_binary(y_test, y_prob, class_names, title):
    """
    Plots the ROC curve for binary classification.
    """
    fpr, tpr, _ = roc_curve(y_test, y_prob[:, 1])
    roc_auc_val = auc(fpr, tpr)

    plt.figure(figsize=(7, 6))
    plt.plot(fpr, tpr, lw=2, color='steelblue',
             label=f'{class_names[1]} (AUC = {roc_auc_val:.4f})')
    plt.plot([0, 1], [0, 1], 'k:', lw=1, label='Random baseline')
    plt.xlim([0, 1])
    plt.ylim([0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'ROC Curve — {title}', fontsize=12)
    plt.legend(loc='lower right', fontsize=10)
    plt.tight_layout()
    plt.savefig(f'../results/roc_{title.replace(" ","_")}.png', dpi=150, bbox_inches='tight')
    plt.show()


plot_roc_curve_binary(
    y_test, final_results['y_prob'], class_names,
    f'Best Stacking Model ({best_name}) — Binary'
)

---
## CELL 22 — Full Thesis-Ready Summary Table

Combines scorecard + final test results into one export-ready table comparing all meta-learners and the final selected model.

In [ ]:
print('\n' + '='*80)
print('  FULL COMPARATIVE SUMMARY — META-LEARNER SELECTION + FINAL TEST RESULTS')
print('='*80)

rows = []
for meta_name, vals in results.items():
    is_selected = (meta_name == best_name)
    rows.append({
        'Model': f'Stacking + {meta_name}',
        'Source': 'This work',
        'CV Acc (%)': round(vals['CV Accuracy Mean'], 2),
        'CV Std (%)': round(vals['Stability (std↓)'], 4),
        'ECE': round(vals['ECE (↓ better)'], 4),
        'Train Time (s)': round(vals['Train Time (s)'], 3),
        'Interpretable': vals['Interpretable'],
        'Test Acc (%)': round(final_results['accuracy'], 2) if is_selected else '(not evaluated)',
        'Test F1 (%)': round(final_results['f1'], 2) if is_selected else '(not evaluated)',
        'Selected': '★ SELECTED' if is_selected else ''
    })

summary_df = pd.DataFrame(rows)
display(summary_df)
summary_df.to_csv('../results/full_summary_binary.csv', index=False)
print('  → Saved to ffull_summary_binary.csv')
print(f'\n  Selected meta-learner: {best_name}')
print(f'  Test Accuracy        : {final_results["accuracy"]:.2f}%')
print(f'  Test F1 (binary)     : {final_results["f1"]:.2f}%')
print(f'  ECE of selected model: {final_results["ece"]:.4f} (test set)')

---
## CELL 23 — Save All Trained Models

In [ ]:
import joblib

# Save best meta-learner
joblib.dump(best_model, f'../results/best_meta_learner_{best_name.replace(" ","_")}.pkl')

# Save base models
joblib.dump(trained_bases, '../results/trained_base_models.pkl')

# Save scaler
joblib.dump(scaler, '../results/scaler.pkl')

print('Models saved:')
print(f'  best_meta_learner_{best_name.replace(" ","_")}.pkl')
print('  trained_base_models.pkl')
print('  scaler.pkl')

---
## Summary Table — Cell Guide

| Cell | Purpose | Uses Test Set? |
|------|---------|----------------|
| 1 | Install packages | No |
| 2 | Import all libraries + global settings | No |
| 3 | Load dataset | No |
| 4 | Preprocessing (encode, clean) | No |
| 5 | Build feature matrix (Binary: Normal vs Theft) | No |
| 6 | Train/test split + StandardScaler | Split only |
| 7 | Define base models | No |
| 8 | Define 5 meta-learner candidates (no Gradient Boosting) | No |
| 9 | Generate OOF meta-features (train only) | **No** |
| 10 | Define ECE calculator | No |
| 11 | CV comparison across all meta-learners | **No** |
| 12 | Friedman statistical test | No |
| 13 | Nemenyi post-hoc pairwise test | No |
| 14 | Calibration ECE bar chart | No |
| 15 | CV accuracy box plots | No |
| 16 | Build scorecard table (highlighted) | No |
| 17 | Radar chart across all criteria | No |
| 18 | Select best meta-learner | No |
| 19 | **Final test set evaluation** | **YES — only here** |
| 20 | Confusion matrix (final model) | Yes (via Cell 19 results) |
| 21 | ROC curve (final model) | Yes (via Cell 19 results) |
| 22 | Full comparative summary table + CSV export | Yes (via Cell 19 results) |
| 23 | Save all models to disk | No |

---

### Thesis-ready paragraph (template)

> Five meta-learner candidates were evaluated to determine the optimal combination strategy for the stacking ensemble: Logistic Regression, Ridge Classifier, Decision Tree, SVM (RBF), and Extra Trees. Selection was performed exclusively on training data using 5-fold stratified cross-validation on out-of-fold meta-features, ensuring no information from the test set influenced model selection. Candidates were assessed across five criteria: CV accuracy, binary F1-score, Expected Calibration Error (ECE), training time, and cross-fold stability. Statistical significance of observed differences was verified using the Friedman test followed by Nemenyi post-hoc pairwise comparison. The selected meta-learner was subsequently evaluated once on the held-out test set to produce the final reported metrics.